In [23]:
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_classic.chains import GraphCypherQAChain
from pinecone import Pinecone
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv()


True

In [24]:
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

graph = Neo4jGraph(
    url="neo4j+s://f5886a71.databases.neo4j.io",
    username="neo4j",
    password="fFCqS5-Ak3ZLppY0p1e2rrL1ZN_FJwt7oH_McxlCh5o"
)

pc = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = "lol-rag"  
pinecone_index = pc.Index(INDEX_NAME)


openai_client = OpenAI(api_key=OPENAI_API_KEY)

llm = ChatOpenAI(
    model_name="gpt-4",
    temperature=0, 
    api_key=OPENAI_API_KEY
)

graph.refresh_schema()


In [25]:
def create_embedding(text):
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


In [26]:
def pinecone_search(query, top_k=10, filter_type=None):
    """
    Semantic search using Pinecone.
    
    Args:
        query: Natural language query
        top_k: Number of results
        filter_type: 'ability' or 'champion' or None
    
    Returns:
        List of matches with metadata
    """
    query_embedding = create_embedding(query)
    
    query_filter = None
    if filter_type:
        query_filter = {'type': filter_type}
    
    results = pinecone_index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        filter=query_filter
    )
    
    return results['matches']

print("✓ Pinecone search function ready")

✓ Pinecone search function ready


In [27]:
# Create GraphCypherQAChain for Neo4j queries
neo4j_chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True,
    return_intermediate_steps=True  # Important: gives us the Cypher query
)

print("✓ Neo4j chain ready")

✓ Neo4j chain ready


In [28]:
# Create GraphCypherQAChain for Neo4j queries
# IMPORTANT: Add custom prompt to enforce Title Case
from langchain_classic.prompts import PromptTemplate

CYPHER_GENERATION_TEMPLATE = """Task: Generate Cypher statement to query a graph database.

Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.

CRITICAL NAMING CONVENTION:
All node property values use Title Case (first letter uppercase):
- Correct: {{name: 'Stun'}}, {{name: 'Grounded'}}, {{name: 'Mage'}}, {{name: 'Magic'}}
- Wrong: {{name: 'stun'}}, {{name: 'grounded'}}, {{name: 'mage'}}, {{name: 'magic'}}

Examples:
- Role names: 'Mage', 'Tank', 'Assassin', 'Support', 'Fighter', 'Marksman'
- Damage types: 'Magic', 'Physical', 'True'
- Crowd control: 'Stun', 'Root', 'Slow', 'Knockup', 'Knockback', 'Grounded', 'Taunt', 'Charm', 'Fear', 'Blind', 'Silence', 'Disarm', 'Suppress'
- Attack types: 'Melee', 'Ranged'
- Stats: 'AP', 'AD', 'Health', 'Armor', 'MagicResist', 'AttackSpeed', 'CriticalStrike'
- Effect types: 'Shield', 'Heal', 'Dash', 'Blink', 'Stealth', 'Movement', 'AttackSpeed', 'Lifesteal', 'Vamp'
- Ability slots: 'Passive', 'Q', 'W', 'E', 'R'

Schema:
{schema}

Question: {question}

Cypher query:"""

CYPHER_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template=CYPHER_GENERATION_TEMPLATE
)

neo4j_chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True,
    return_intermediate_steps=True,
    cypher_prompt=CYPHER_PROMPT  # Use our custom prompt
)

print("✓ Neo4j chain ready (with Title Case enforcement)")

✓ Neo4j chain ready (with Title Case enforcement)


In [29]:
def direct_neo4j_query(cypher_query, params=None):
    """
    Execute a direct Cypher query (bypasses GPT-4 chain).
    
    Use this when you want full control over the query.
    
    Args:
        cypher_query: Cypher query string
        params: Optional query parameters
    
    Returns:
        List of result dictionaries
    """
    try:
        results = graph.query(cypher_query, params=params or {})
        return results
    except Exception as e:
        print(f"❌ Query failed: {e}")
        return []

def fix_cypher_case(cypher_query):
    """
    Auto-fix common capitalization errors in Cypher queries.
    
    Converts lowercase node property values to Title Case.
    This is a safety net in case GPT-4 generates wrong capitalization.
    
    Args:
        cypher_query: Cypher query string (possibly with wrong case)
    
    Returns:
        Fixed Cypher query string
    """
    import re
    
    # Common values that should be Title Case
    replacements = {
        # Roles
        r"\{name: '(mage|tank|assassin|support|fighter|marksman)'": lambda m: f"{{name: '{m.group(1).capitalize()}'",
        # Damage types
        r"\{name: '(magic|physical|true)'": lambda m: f"{{name: '{m.group(1).capitalize()}'",
        # Crowd control (most important - this is what failed!)
        r"\{name: '(stun|root|slow|knockup|knockback|grounded|taunt|charm|fear|blind|silence|disarm|suppress)'": lambda m: f"{{name: '{m.group(1).capitalize()}'",
        # Attack types
        r"\{name: '(melee|ranged)'": lambda m: f"{{name: '{m.group(1).capitalize()}'",
        # Stats (special - AP/AD stay uppercase)
        r"\{name: '(ap|ad)'": lambda m: f"{{name: '{m.group(1).upper()}'",
        r"\{name: '(health|armor)'": lambda m: f"{{name: '{m.group(1).capitalize()}'",
    }
    
    fixed_query = cypher_query
    for pattern, replacement in replacements.items():
        fixed_query = re.sub(pattern, replacement, fixed_query, flags=re.IGNORECASE)
    
    return fixed_query

print("✓ Direct Neo4j query function ready")
print("✓ Cypher case-fixing function ready")

✓ Direct Neo4j query function ready
✓ Cypher case-fixing function ready


In [30]:
def analyze_query(question):
    """
    Analyze user question and determine retrieval strategy.
    
    Returns:
        dict with:
        - strategy: 'neo4j', 'pinecone', or 'hybrid'
        - reasoning: Why this strategy
        - neo4j_hints: Suggested Cypher patterns (if applicable)
        - semantic_query: Reformulated query for Pinecone (if applicable)
    """
    
    analysis_prompt = f"""
You are a query analyzer for a League of Legends knowledge base with two databases:

**Neo4j (Graph Database):**
- Structured data: roles, damage types, crowd control, scaling stats
- Good for: exact filters, boolean logic, relationships
- Schema: {graph.schema[:800]}

**Pinecone (Vector Database):**
- Semantic search on ability descriptions and champion profiles
- Good for: conceptual queries, themes, similarity
- Examples: "fire abilities", "tanky champions", "burst damage"

Analyze this question: "{question}"

Respond in JSON format:
{{
    "strategy": "neo4j" | "pinecone" | "hybrid",
    "reasoning": "why you chose this strategy",
    "neo4j_hints": "suggested concepts to query" (if using neo4j),
    "semantic_query": "reformulated query for vector search" (if using pinecone),
    "filters": {{
        "role": "Mage" (example),
        "attack_type": "Ranged" (example)
    }} (if using hybrid)
}}

**Decision Rules:**
- Use Neo4j for: specific roles, attack types, exact damage/CC types, difficulty
- Use Pinecone for: themes (fire/ice), playstyles (burst/poke), vague concepts
- Use Hybrid for: combinations ("ranged mages with fire")
"""
    
    response = openai_client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a query strategy analyzer. Always respond with valid JSON."},
            {"role": "user", "content": analysis_prompt}
        ],
        temperature=0
    )
    
    # Parse JSON response
    try:
        analysis = json.loads(response.choices[0].message.content)
        return analysis
    except json.JSONDecodeError:
        # Fallback if JSON parsing fails
        return {
            "strategy": "neo4j",
            "reasoning": "Default fallback",
            "neo4j_hints": question
        }

print("✓ Query analyzer ready")

✓ Query analyzer ready


In [31]:
def retrieve_context(question):
    """
    Master retrieval function.
    
    1. Analyze query
    2. Retrieve from appropriate source(s)
    3. Format context for GPT-4
    
    Returns:
        dict with:
        - context: Formatted text for GPT-4
        - sources: List of sources used
        - strategy: Which strategy was used
    """
    
    # Step 1: Analyze query
    analysis = analyze_query(question)
    strategy = analysis['strategy']
    
    print(f"\n🔍 Strategy: {strategy}")
    print(f"   Reasoning: {analysis['reasoning']}")
    
    context_parts = []
    sources = []
    
    # Step 2: Execute retrieval based on strategy
    
    if strategy == 'neo4j':
        # Pure Neo4j query
        print("\n📊 Querying Neo4j...")
        result = neo4j_chain.invoke({"query": question})
        
        intermediate = result.get('intermediate_steps', [])

        if len(intermediate) >= 2:
            # Get the actual Cypher query and its raw results
            cypher_query = intermediate[0].get('query', 'N/A')
            raw_results = intermediate[1].get('context', [])
            
            print(f"   Cypher: {cypher_query[:80]}...")
            print(f"   Found: {len(raw_results)} results")
            
            # FALLBACK: If empty results, try fixing capitalization
            if not raw_results and 'name:' in cypher_query.lower():
                print("   ⚠️ No results - trying with case correction...")
                fixed_query = fix_cypher_case(cypher_query)
                if fixed_query != cypher_query:
                    print(f"   Fixed: {fixed_query[:80]}...")
                    raw_results = graph.query(fixed_query)
                    cypher_query = fixed_query  # Update for display
                    print(f"   Found: {len(raw_results)} results after fix")
            
            # Format the RAW data for GPT-4 (not the chain's answer)
            if raw_results:
                context_parts.append(f"**Neo4j Query Results:**")
                context_parts.append(f"\nCypher Query: {cypher_query}")
                context_parts.append(f"\nResults ({len(raw_results)} found):\n")
                
                # Format each result row
                for i, row in enumerate(raw_results[:50], 1):  # Limit to 50
                    # Convert dict to readable format
                    formatted_row = ", ".join([f"{k}: {v}" for k, v in row.items()])
                    context_parts.append(f"{i}. {formatted_row}")
                
                sources.append(f"Neo4j: {len(raw_results)} results")
            else:
                context_parts.append(f"**Neo4j Query:**\n{cypher_query}\n\nNo results found.")
                sources.append("Neo4j: No results")
        else:
            # Fallback: use chain's answer if intermediate steps unavailable
            context_parts.append(f"**Neo4j Results:**\n{result['result']}")
            sources.append("Neo4j Query")
    
    elif strategy == 'pinecone':
        # Pure semantic search
        semantic_query = analysis.get('semantic_query', question)
        print(f"\n🔍 Searching Pinecone: '{semantic_query}'")
        
        matches = pinecone_search(semantic_query, top_k=10)
        
        # Format results
        context_parts.append("**Semantic Search Results:**")
        for i, match in enumerate(matches[:5], 1):
            meta = match['metadata']
            if meta['type'] == 'ability':
                context_parts.append(
                    f"{i}. {meta['champion_name']} - {meta['ability_name']} [{meta['slot']}]\n"
                    f"   Preview: {meta['text'][:150]}..."
                )
            else:
                context_parts.append(
                    f"{i}. {meta['champion_name']} - {meta['title']}\n"
                    f"   Roles: {meta.get('roles', 'N/A')}"
                )
            sources.append(f"Pinecone: {meta['champion_name']}")
    
    elif strategy == 'hybrid':
        # Combined approach
        print("\n🔀 Hybrid Retrieval (Neo4j + Pinecone)")
        
        # First, semantic search
        semantic_query = analysis.get('semantic_query', question)
        print(f"   1. Pinecone: '{semantic_query}'")
        matches = pinecone_search(semantic_query, top_k=15)
        
        # Extract champion IDs from semantic results
        champion_ids = list(set([m['metadata']['champion_id'] for m in matches]))
        
        # Second, filter with Neo4j
        filters = analysis.get('filters', {})
        if filters:
            print(f"   2. Neo4j filters: {filters}")
            
            # Build Cypher query
            cypher_parts = ["MATCH (c:Champion) WHERE c.id IN $champion_ids"]
            params = {'champion_ids': champion_ids}
            
            if 'role' in filters:
                cypher_parts.append("MATCH (c)-[:HAS_ROLE]->(r:Role {name: $role})")
                params['role'] = filters['role']
            
            if 'attack_type' in filters:
                cypher_parts.append("MATCH (c)-[:ATTACKS_WITH]->(at:AttackType {name: $attack_type})")
                params['attack_type'] = filters['attack_type']
            
            cypher_parts.append("RETURN DISTINCT c.id as id, c.name as name")
            cypher_query = " ".join(cypher_parts)
            
            filtered_champions = graph.query(cypher_query, params=params)
            filtered_ids = {c['id'] for c in filtered_champions}
            
            # Filter Pinecone results
            matches = [m for m in matches if m['metadata']['champion_id'] in filtered_ids]
        
        # Format combined results
        context_parts.append("**Hybrid Search Results:**")
        for i, match in enumerate(matches[:5], 1):
            meta = match['metadata']
            if meta['type'] == 'ability':
                context_parts.append(
                    f"{i}. {meta['champion_name']} - {meta['ability_name']} [{meta['slot']}]\n"
                    f"   Similarity: {match['score']:.3f}\n"
                    f"   {meta['text'][:150]}..."
                )
            sources.append(f"Hybrid: {meta['champion_name']}")
    
    # Step 3: Assemble final context
    context = "\n\n".join(context_parts)
    
    return {
        'context': context,
        'sources': sources[:5],  # Limit sources
        'strategy': strategy,
        'analysis': analysis
    }

print("✓ Master retrieval function ready")

✓ Master retrieval function ready


In [32]:
def generate_answer(question, context_data):
    """
    Generate final answer using GPT-4.
    
    Args:
        question: Original user question
        context_data: Retrieved context from retrieve_context()
    
    Returns:
        Natural language answer
    """
    
    system_prompt = """
You are a League of Legends expert assistant.

Your job:
1. Answer questions accurately using the provided context
2. Be concise but informative
3. Cite specific champions/abilities when relevant
4. If context is insufficient, say so honestly
5. Use natural, conversational language

Guidelines:
- Don't make up information not in the context
- If asked about specific numbers (damage, cooldowns), only provide if in context
- Format lists clearly when showing multiple results
- Explain WHY champions/abilities match the query
"""
    
    user_prompt = f"""
Question: {question}

Context Retrieved:
{context_data['context']}

Please provide a clear, accurate answer based on this context.
"""
    
    response = openai_client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3  # Slightly creative but mostly factual
    )
    
    return response.choices[0].message.content

print("✓ Answer generator ready")

✓ Answer generator ready


In [33]:
def ask_question(question, verbose=True):
    """
    Main Q&A function - handles everything end-to-end.
    
    Args:
        question: Natural language question
        verbose: Show retrieval details
    
    Returns:
        Answer string
    """
    
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")
    
    # Retrieve context
    context_data = retrieve_context(question)
    
    if verbose:
        print(f"\n📦 Context retrieved ({len(context_data['context'])} chars)")
        print(f"   Strategy: {context_data['strategy']}")
        print(f"   Sources: {', '.join(context_data['sources'][:3])}...")
    
    # Generate answer
    print(f"\n🤖 Generating answer...")
    answer = generate_answer(question, context_data)
    
    print(f"\n{'='*60}")
    print(f"💡 ANSWER:")
    print(f"{'='*60}")
    print(answer)
    print(f"{'='*60}\n")
    
    return {
        'question': question,
        'answer': answer,
        'strategy': context_data['strategy'],
        'sources': context_data['sources']
    }

print("✓ Main Q&A function ready")
print("\n🚀 Ready to answer questions!")
print("\nUsage: ask_question('Find ranged mages with crowd control')")

✓ Main Q&A function ready

🚀 Ready to answer questions!

Usage: ask_question('Find ranged mages with crowd control')


In [43]:
ask_question("What's Annie's crowd control ability?")


❓ Question: What's Annie's crowd control ability?

🔍 Strategy: neo4j
   Reasoning: The query is asking for a specific champion's ability, which is a structured data query that Neo4j is well-suited for.

📊 Querying Neo4j...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Champion {name: 'Annie'})-[:HAS_ABILITY]->(a:Ability)-[:APPLIES]->(cc:CrowdControl)
RETURN a.name, cc.name
Full Context:
[{'a.name': 'Pyromania', 'cc.name': 'Stun'}, {'a.name': 'Summon: Tibbers', 'cc.name': 'Stun'}]

> Finished chain.
   Cypher: MATCH (c:Champion {name: 'Annie'})-[:HAS_ABILITY]->(a:Ability)-[:APPLIES]->(cc:C...
   Found: 2 results

📦 Context retrieved (258 chars)
   Strategy: neo4j
   Sources: Neo4j: 2 results...

🤖 Generating answer...

💡 ANSWER:
Annie has two abilities that apply crowd control:

1. Pyromania: This ability stuns enemies.
2. Summon: Tibbers: This ability also stuns enemies.



{'question': "What's Annie's crowd control ability?",
 'answer': 'Annie has two abilities that apply crowd control:\n\n1. Pyromania: This ability stuns enemies.\n2. Summon: Tibbers: This ability also stuns enemies.',
 'strategy': 'neo4j',
 'sources': ['Neo4j: 2 results']}